## Load Visual Groundtruth Data

In [1]:
import pandas as pd
import numpy as np
import warnings

import os

import utils
from scipy.spatial import distance

In [2]:
def load_img_groundtruth(root_dir, place, building, floor):
    groundtruth_csv = f'{root_dir}/{place}/{building}/{floor}/groundtruth_img_dataset_{building}_{floor}.csv'
    return pd.read_csv(groundtruth_csv)

In [3]:
root_dir = '/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data'
place = 'Mahidol_University'
building = 'ICT'
floor = '1.1_MixVPR'
groundtruth_df = load_img_groundtruth(root_dir, place, building, floor)

In [4]:
groundtruth_df

,Unnamed: 0,image_name,cx,cy,ang
0,0,000000_pitch00_yaw00,2035.798188,602.251145,67.507674
1,1,000000_pitch00_yaw01,2035.798188,602.251145,87.333218
2,2,000000_pitch00_yaw02,2035.798188,602.251145,107.086629
3,3,000000_pitch00_yaw03,2035.798188,602.251145,126.881578
4,4,000000_pitch00_yaw04,2035.798188,602.251145,146.813368
...,...,...,...,...,...
6025,6025,000336_pitch00_yaw13,2047.409344,636.375384,349.790898
6026,6026,000336_pitch00_yaw14,2047.409344,636.375384,10.045827
6027,6027,000336_pitch00_yaw15,2047.409344,636.375384,30.291353
6028,6028,000336_pitch00_yaw16,2047.409344,636.375384,50.411187


## Load Mag batched data

In [5]:
import pandas as pd
ofn = 'data/ict_fl1/oneplus5t/24dec24/batched_coord_raw_mag_24dec24.h5'
mag_batch_df = pd.read_hdf(ofn)

In [6]:
16200/50

324.0

## Align Visual and Mag Data

In [22]:
def align_mag_img_groundtruth(mag_gt_df, img_gt_df, 
                                       ref_points,
                              distance_threshold_meter=1, pixel_to_meter_factor=27.4):
    r = []
    for cxcy in ref_points:
        cx = float(cxcy.split(',')[0])
        cy = float(cxcy.split(',')[1])
        dists = [ (distance.euclidean([cx,cy], [float(gx), float(gy)]), img_name) for (gx,gy,img_name) in img_gt_df[['cx', 'cy', 'image_name']].itertuples(index=False) ]
        nearby = [ (d, img_name) for d, img_name in dists if d/pixel_to_meter_factor <= distance_threshold_meter ]
        r += [{
            'image_name': i,
            'Cx': cx,
            'Cy': cy,
            'dist':d,
            } for d, i in nearby
        ]
    return pd.DataFrame.from_records( r )

In [23]:

mag_batch_df['label'] = mag_batch_df['Cx'].astype(str) +','+ mag_batch_df['Cy'].astype(str)
ref_points = mag_batch_df['label'].unique()
aligned_df = align_mag_img_groundtruth(mag_batch_df, 
                                       groundtruth_df, 
                                       ref_points,
                                       distance_threshold_meter=1, 
                                       pixel_to_meter_factor=27.4)
aligned_df

,image_name,Cx,Cy,dist
0,000034_pitch00_yaw00,1920.0,1330.0,13.054279
1,000034_pitch00_yaw01,1920.0,1330.0,13.054279
2,000034_pitch00_yaw02,1920.0,1330.0,13.054279
3,000034_pitch00_yaw03,1920.0,1330.0,13.054279
4,000034_pitch00_yaw04,1920.0,1330.0,13.054279
...,...,...,...,...
283,000079_pitch00_yaw13,846.0,1324.0,25.428418
284,000079_pitch00_yaw14,846.0,1324.0,25.428418
285,000079_pitch00_yaw15,846.0,1324.0,25.428418
286,000079_pitch00_yaw16,846.0,1324.0,25.428418


## Segment

In [27]:

def get_num_stratified_images(aligned_df):
    aligned_df['label'] = aligned_df['Cx'].astype(str) +','+ aligned_df['Cy'].astype(str)
    return min([len(aligned_df[ aligned_df['label'] == l ]) for l in aligned_df['label'].unique()])

def select_segmented_mag_data_for_image_i(idx, num_batches_per_stratified_image, mag_batches):
    '''
    Returns segmented batches between index start and end.
    length, index_start, index_end of mag_batches
    9 0 9
    9 9 18
    9 18 27
    9 27 36
    9 36 45
    9 45 54
    9 54 63
    9 63 72
    '''
    idx_start = idx*num_batches_per_stratified_image
    idx_end = (idx*num_batches_per_stratified_image) + num_batches_per_stratified_image
    return mag_batches[idx_start:idx_end], idx_start

def segment(mag_gt_df, ref_points, aligned_df, seconds=1, hz=50):
    '''
    '''
    mag_records_per_ref_point = len(mag_gt_df) / len(ref_points) # e.g. 16200
    distinct_aligned_ref_points = aligned_df[ ['Cx', 'Cy'] ].drop_duplicates() # e.g. 6
    r = []
    for row in distinct_aligned_ref_points.iterrows():
        cx, cy = row[1]['Cx'], row[1]['Cy']
        dft = mag_batch_df[mag_batch_df['label']==f"{cx},{cy}"]
        if len(dft) != mag_records_per_ref_point:
            raise ArgumentError(f'Magnetometer data is not stratified (balanced/equal) across all ref points.\nProblem with ref point: {cx},{cy}.')
        
        images = aligned_df[ (aligned_df ['Cx'] == cx) & (aligned_df['Cy'] == cy) ]['image_name'].to_list()
        mag_batches = np.array(np.array_split(dft[ ['X', 'Y', 'Z'] ], len(dft) // (seconds*hz)))
        # Get the number of images
        min_balanced_aligned_img = get_num_stratified_images(aligned_df) # 36, not 72.
        stratified_images = images[:min_balanced_aligned_img]
        equally_segmented_size = len(stratified_images)
        num_batches_per_stratified_image = len(mag_batches)//equally_segmented_size
        # Distribute mag data evenly between images
        for idx, i in enumerate(stratified_images):
            # select 9 mag data 
            mag_data, idx_start = select_segmented_mag_data_for_image_i(idx, num_batches_per_stratified_image, mag_batches)
            for j, m in enumerate(mag_data):
                # r += [ [i, idx_start+j, cx, cy ] ]
                for k in m:
                    r += [ {'image_name':i, 
                        'Mx':k[0],
                        'My':k[1],
                        'Mz':k[2],
                        'Cx':cx, 
                        'Cy':cy } ]
    return r
    

In [28]:
pair = segment(mag_batch_df,  ref_points, aligned_df, seconds=1, hz=50)


/unav/venv/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
/unav/venv/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
/unav/venv/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
/unav/venv/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
/unav/venv/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59

In [29]:
pd.DataFrame.from_records(pair)

,image_name,Mx,My,Mz,Cx,Cy
0,000034_pitch00_yaw00,-2.098846,29.233932,-23.312187,1920.0,1330.0
1,000034_pitch00_yaw00,-2.098846,29.983521,-24.061775,1920.0,1330.0
2,000034_pitch00_yaw00,-1.349258,29.233932,-23.312187,1920.0,1330.0
3,000034_pitch00_yaw00,-3.672981,29.233932,-24.061775,1920.0,1330.0
4,000034_pitch00_yaw00,-1.349258,29.233932,-22.637558,1920.0,1330.0
...,...,...,...,...,...,...
97195,000079_pitch00_yaw17,-14.841843,-23.686981,15.066719,846.0,1324.0
97196,000079_pitch00_yaw17,-14.092255,-25.186157,15.066719,846.0,1324.0
97197,000079_pitch00_yaw17,-14.841843,-23.686981,16.565895,846.0,1324.0
97198,000079_pitch00_yaw17,-12.518120,-24.436569,17.315483,846.0,1324.0


## Saving

In [12]:
def save_aligned_data(aligned_df, root_dir, place, building, floor):
    out_fn = f'{root_dir}/{place}/{building}/{floor}/Aligned_Mag_Img_{building}_{floor}.h5'
    aligned_df.to_hdf(out_fn, key="data", mode="w")
    print(f"Aligned data was saved to {out_fn}.")

In [30]:
save_aligned_data(aligned_df, root_dir, place, building, floor)

Aligned data was saved to /home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_MixVPR/Aligned_Mag_Img_ICT_1.1_MixVPR.h5.
